In [5]:
import pandas as pd

print(pd.__version__)

3.1.1


In [7]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd

df = pd.read_csv("../02_Datasets/Nexora_Sales_Dataset.csv")

print(df.head())

  Opportunity_ID  Lead_ID Sales_Representative Opportunity_Stage  \
0         OPP001  LEAD001                 Asha     Qualification   
1         OPP002  LEAD002                Rahul          Proposal   
2         OPP003  LEAD003                Priya        Closed Won   
3         OPP004  LEAD004                Arjun       Negotiation   
4         OPP005  LEAD005                Meera       Closed Lost   

   Opportunity_Amount Opportunity_Status Last_Activity_Date  \
0              250000               Open         2026-10-25   
1              450000               Open         2026-09-15   
2              600000                Won         2026-10-12   
3              800000               Open         2026-10-28   
4              300000               Lost         2026-09-20   

  Next_Followup_Date Expected_Close_Date Actual_Close_Date Lead_Source  \
0         2026-11-08          2026-11-30               NaN     Website   
1         2026-09-22          2026-10-20               NaN    Re

In [9]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

Dataset Shape: (30, 13)

Column Names:
['Opportunity_ID', 'Lead_ID', 'Sales_Representative', 'Opportunity_Stage', 'Opportunity_Amount', 'Opportunity_Status', 'Last_Activity_Date', 'Next_Followup_Date', 'Expected_Close_Date', 'Actual_Close_Date', 'Lead_Source', 'Lead_Creation_Date', 'Lead_Status']

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Opportunity_ID        30 non-null     str  
 1   Lead_ID               30 non-null     str  
 2   Sales_Representative  30 non-null     str  
 3   Opportunity_Stage     30 non-null     str  
 4   Opportunity_Amount    30 non-null     int64
 5   Opportunity_Status    30 non-null     str  
 6   Last_Activity_Date    30 non-null     str  
 7   Next_Followup_Date    20 non-null     str  
 8   Expected_Close_Date   30 non-null     str  
 9   Actual_Close_Date     10 non-null     str  


In [10]:
print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Missing Values:
Opportunity_ID           0
Lead_ID                  0
Sales_Representative     0
Opportunity_Stage        0
Opportunity_Amount       0
Opportunity_Status       0
Last_Activity_Date       0
Next_Followup_Date      10
Expected_Close_Date      0
Actual_Close_Date       20
Lead_Source              0
Lead_Creation_Date       0
Lead_Status              0
dtype: int64

Duplicate Rows:
0


In [11]:
print(
    df.groupby("Opportunity_Status")[
        ["Next_Followup_Date", "Actual_Close_Date"]
    ].apply(lambda x: x.isnull().sum())
)

                    Next_Followup_Date  Actual_Close_Date
Opportunity_Status                                       
Lost                                 4                  0
Open                                 0                 20
Won                                  6                  0


In [12]:
date_columns = [
    "Last_Activity_Date",
    "Next_Followup_Date",
    "Expected_Close_Date",
    "Actual_Close_Date",
    "Lead_Creation_Date"
]

print(df[date_columns].head(5))

  Last_Activity_Date Next_Followup_Date Expected_Close_Date Actual_Close_Date  \
0         2026-10-25         2026-11-08          2026-11-30               NaN   
1         2026-09-15         2026-09-22          2026-10-20               NaN   
2         2026-10-12                NaN          2026-10-15        2026-10-12   
3         2026-10-28         2026-11-05          2026-11-25               NaN   
4         2026-09-20                NaN          2026-09-30        2026-09-20   

  Lead_Creation_Date  
0         2026-09-12  
1         2026-08-10  
2         2026-07-05  
3         2026-09-01  
4         2026-07-18  


In [13]:
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[date_columns].dtypes)

Last_Activity_Date     datetime64[us]
Next_Followup_Date     datetime64[us]
Expected_Close_Date    datetime64[us]
Actual_Close_Date      datetime64[us]
Lead_Creation_Date     datetime64[us]
dtype: object


In [14]:
REPORT_DATE = pd.Timestamp("2026-10-31")

print("Reporting Date:", REPORT_DATE)
print("Total Records:", len(df))
print("Total Columns:", len(df.columns))

Reporting Date: 2026-10-31 00:00:00
Total Records: 30
Total Columns: 13


In [15]:
print("Amount Data Type:", df["Opportunity_Amount"].dtype)

print("\nTotal Opportunity Value:")
print(df["Opportunity_Amount"].sum())

print("\nMinimum Amount:")
print(df["Opportunity_Amount"].min())

print("\nMaximum Amount:")
print(df["Opportunity_Amount"].max())

Amount Data Type: int64

Total Opportunity Value:
16550000

Minimum Amount:
180000

Maximum Amount:
1200000


In [16]:
print(df["Opportunity_Status"].value_counts())

print("\nStatus Percentages:")
print(df["Opportunity_Status"].value_counts(normalize=True) * 100)

Opportunity_Status
Open    20
Won      6
Lost     4
Name: count, dtype: int64

Status Percentages:
Opportunity_Status
Open    66.666667
Won     20.000000
Lost    13.333333
Name: proportion, dtype: float64


In [17]:
status_analysis = df.groupby("Opportunity_Status").agg(
    Opportunity_Count=("Opportunity_ID", "count"),
    Total_Amount=("Opportunity_Amount", "sum")
)

print(status_analysis)

                    Opportunity_Count  Total_Amount
Opportunity_Status                                 
Lost                                4       1600000
Open                               20      10580000
Won                                 6       4370000


In [18]:
overdue_df = df[
    (df["Opportunity_Status"] == "Open") &
    (df["Next_Followup_Date"] < REPORT_DATE)
]

print("Overdue Opportunities:", len(overdue_df))

print("Total Overdue Pipeline Value:",
      overdue_df["Opportunity_Amount"].sum())

Overdue Opportunities: 12
Total Overdue Pipeline Value: 6390000


In [19]:
STALE_CUTOFF = pd.Timestamp("2026-10-01")

stale_df = df[
    (df["Opportunity_Status"] == "Open") &
    (df["Last_Activity_Date"] < STALE_CUTOFF)
]

print("Stale Opportunities:", len(stale_df))

print("Total Stale Pipeline Value:",
      stale_df["Opportunity_Amount"].sum())

Stale Opportunities: 10
Total Stale Pipeline Value: 5080000


In [20]:
overdue_ids = set(overdue_df["Opportunity_ID"])
stale_ids = set(stale_df["Opportunity_ID"])

print("Overdue:", len(overdue_ids))
print("Stale:", len(stale_ids))
print("Both Overdue and Stale:", len(overdue_ids & stale_ids))
print("Unique Risk Opportunities:", len(overdue_ids | stale_ids))

Overdue: 12
Stale: 10
Both Overdue and Stale: 10
Unique Risk Opportunities: 12


In [21]:
import numpy as np

conditions = [
    (df["Opportunity_ID"].isin(overdue_ids) & df["Opportunity_ID"].isin(stale_ids)),
    (df["Opportunity_ID"].isin(overdue_ids) & ~df["Opportunity_ID"].isin(stale_ids)),
    (~df["Opportunity_ID"].isin(overdue_ids) & df["Opportunity_ID"].isin(stale_ids))
]

categories = [
    "Overdue and Stale",
    "Overdue Only",
    "Stale Only"
]

df["Risk_Category"] = np.select(
    conditions,
    categories,
    default="No Risk Flag"
)

risk_analysis = df.groupby("Risk_Category").agg(
    Opportunity_Count=("Opportunity_ID", "count"),
    Pipeline_Value=("Opportunity_Amount", "sum")
)

print(risk_analysis)

                   Opportunity_Count  Pipeline_Value
Risk_Category                                       
No Risk Flag                      18        10160000
Overdue Only                       2         1310000
Overdue and Stale                 10         5080000


In [22]:
open_risk_analysis = df[
    df["Opportunity_Status"] == "Open"
].groupby("Risk_Category").agg(
    Opportunity_Count=("Opportunity_ID", "count"),
    Pipeline_Value=("Opportunity_Amount", "sum")
)

print(open_risk_analysis)

                   Opportunity_Count  Pipeline_Value
Risk_Category                                       
No Risk Flag                       8         4190000
Overdue Only                       2         1310000
Overdue and Stale                 10         5080000


In [23]:
rep_overdue = overdue_df.groupby("Sales_Representative").agg(
    Overdue_Count=("Opportunity_ID", "count"),
    Overdue_Value=("Opportunity_Amount", "sum")
).sort_values("Overdue_Count", ascending=False)

print(rep_overdue)

                      Overdue_Count  Overdue_Value
Sales_Representative                              
Asha                              3        1650000
Rahul                             3        1540000
Meera                             3        1320000
Priya                             2        1180000
Arjun                             1         700000


In [24]:
rep_total_open = df[df["Opportunity_Status"] == "Open"].groupby(
    "Sales_Representative"
).size()

rep_overdue["Total_Open"] = rep_total_open

rep_overdue["Overdue_Rate"] = (
    rep_overdue["Overdue_Count"] /
    rep_overdue["Total_Open"] * 100
)

print(rep_overdue)

                      Overdue_Count  Overdue_Value  Total_Open  Overdue_Rate
Sales_Representative                                                        
Asha                              3        1650000           5     60.000000
Rahul                             3        1540000           4     75.000000
Meera                             3        1320000           4     75.000000
Priya                             2        1180000           3     66.666667
Arjun                             1         700000           4     25.000000


In [25]:
stage_analysis = df[
    df["Opportunity_Status"] == "Open"
].groupby("Opportunity_Stage").agg(
    Opportunity_Count=("Opportunity_ID", "count"),
    Pipeline_Value=("Opportunity_Amount", "sum")
)

stage_analysis["Pipeline_Share"] = (
    stage_analysis["Pipeline_Value"] /
    stage_analysis["Pipeline_Value"].sum() * 100
)

print(stage_analysis)

                   Opportunity_Count  Pipeline_Value  Pipeline_Share
Opportunity_Stage                                                   
Needs Analysis                     5         1710000       16.162571
Negotiation                        5         4500000       42.533081
Proposal                           6         3500000       33.081285
Qualification                      4          870000        8.223062


In [26]:
past_due_close = df[
    (df["Opportunity_Status"] == "Open") &
    (df["Expected_Close_Date"] < REPORT_DATE)
]

print("Past Due Opportunities:", len(past_due_close))

print("Past Due Pipeline Value:",
      past_due_close["Opportunity_Amount"].sum())

Past Due Opportunities: 9
Past Due Pipeline Value: 4610000


In [27]:
past_due_ids = set(past_due_close["Opportunity_ID"])

print("Past Due Close Opportunities:", len(past_due_ids))

print("Both Overdue and Past Due Close:",
      len(overdue_ids & past_due_ids))

print("Combined Unique Opportunities:",
      len(overdue_ids | past_due_ids))

Past Due Close Opportunities: 9
Both Overdue and Past Due Close: 9
Combined Unique Opportunities: 12


In [28]:
closed_df = df[df["Opportunity_Status"].isin(["Won", "Lost"])]

won_count = (closed_df["Opportunity_Status"] == "Won").sum()
closed_count = len(closed_df)

win_rate = (won_count / closed_count) * 100

print("Closed Opportunities:", closed_count)
print("Won Opportunities:", won_count)
print("Win Rate:", round(win_rate, 2), "%")

Closed Opportunities: 10
Won Opportunities: 6
Win Rate: 60.0 %


In [29]:
priority_opportunities = overdue_df.sort_values(
    "Opportunity_Amount",
    ascending=False
)

print(
    priority_opportunities[
        [
            "Opportunity_ID",
            "Sales_Representative",
            "Opportunity_Stage",
            "Opportunity_Amount",
            "Last_Activity_Date",
            "Next_Followup_Date"
        ]
    ].head(5)
)

   Opportunity_ID Sales_Representative Opportunity_Stage  Opportunity_Amount  \
29         OPP030                Meera       Negotiation              900000   
27         OPP028                Priya          Proposal              760000   
15         OPP016                 Asha       Negotiation              750000   
8          OPP009                Arjun          Proposal              700000   
21         OPP022                Rahul          Proposal              620000   

   Last_Activity_Date Next_Followup_Date  
29         2026-07-18         2026-07-25  
27         2026-10-02         2026-10-12  
15         2026-09-12         2026-09-20  
8          2026-07-10         2026-07-18  
21         2026-08-20         2026-08-28  


In [30]:
print(
    df[df["Opportunity_ID"] == "OPP030"].T
)

                                       29
Opportunity_ID                     OPP030
Lead_ID                           LEAD030
Sales_Representative                Meera
Opportunity_Stage             Negotiation
Opportunity_Amount                 900000
Opportunity_Status                   Open
Last_Activity_Date    2026-07-18 00:00:00
Next_Followup_Date    2026-07-25 00:00:00
Expected_Close_Date   2026-09-30 00:00:00
Actual_Close_Date                     NaT
Lead_Source                      Campaign
Lead_Creation_Date    2026-06-12 00:00:00
Lead_Status                     Converted
Risk_Category           Overdue and Stale


In [31]:
print("Total Records:", len(df))
print("Unique Opportunity IDs:", df["Opportunity_ID"].nunique())
print("Duplicate Records:", df.duplicated().sum())

print("\nStatus Reconciliation:")
print(df["Opportunity_Status"].value_counts())

print("\nTotal Opportunity Value:", df["Opportunity_Amount"].sum())

print("\nOpen Pipeline:",
      df.loc[df["Opportunity_Status"] == "Open",
             "Opportunity_Amount"].sum())

Total Records: 30
Unique Opportunity IDs: 30
Duplicate Records: 0

Status Reconciliation:
Opportunity_Status
Open    20
Won      6
Lost     4
Name: count, dtype: int64

Total Opportunity Value: 16550000

Open Pipeline: 10580000


In [32]:
python_summary = pd.DataFrame({
    "Metric": [
        "Total Opportunities",
        "Open Opportunities",
        "Won Opportunities",
        "Lost Opportunities",
        "Total Opportunity Value",
        "Open Pipeline",
        "Closed Won Revenue",
        "Overdue Opportunities",
        "Overdue Pipeline Value",
        "Stale Opportunities",
        "Stale Pipeline Value",
        "Past Due Close Opportunities",
        "Unique Risk Opportunities",
        "Closed Won Rate"
    ],
    "Value": [
        len(df),
        len(df[df["Opportunity_Status"] == "Open"]),
        len(df[df["Opportunity_Status"] == "Won"]),
        len(df[df["Opportunity_Status"] == "Lost"]),
        df["Opportunity_Amount"].sum(),
        df.loc[df["Opportunity_Status"] == "Open", "Opportunity_Amount"].sum(),
        df.loc[df["Opportunity_Status"] == "Won", "Opportunity_Amount"].sum(),
        len(overdue_df),
        overdue_df["Opportunity_Amount"].sum(),
        len(stale_df),
        stale_df["Opportunity_Amount"].sum(),
        len(past_due_close),
        len(overdue_ids | stale_ids),
        round(win_rate, 2)
    ]
})

print(python_summary.to_string(index=False))

                      Metric      Value
         Total Opportunities       30.0
          Open Opportunities       20.0
           Won Opportunities        6.0
          Lost Opportunities        4.0
     Total Opportunity Value 16550000.0
               Open Pipeline 10580000.0
          Closed Won Revenue  4370000.0
       Overdue Opportunities       12.0
      Overdue Pipeline Value  6390000.0
         Stale Opportunities       10.0
        Stale Pipeline Value  5080000.0
Past Due Close Opportunities        9.0
   Unique Risk Opportunities       12.0
             Closed Won Rate       60.0


## Python Analysis – Key Business Findings

### 1. Dataset Overview
- Total opportunities: 30
- Open: 20
- Won: 6
- Lost: 4
- Total opportunity value: ₹1,65,50,000
- Current open pipeline: ₹1,05,80,000

### 2. Opportunity Risk Analysis
- Overdue follow-ups: 12 opportunities, ₹63.90 lakh
- Stale opportunities: 10 opportunities, ₹50.80 lakh
- Unique risk opportunities: 12
- Overdue and stale: 10 opportunities
- Overdue only: 2 opportunities
- No risk flag: 8 open opportunities
- Past expected close dates: 9 opportunities, ₹46.10 lakh

Risk values represent associated open pipeline, not confirmed lost revenue. Overlapping categories must not be added together.

### 3. Sales Performance
- Closed-won opportunities: 6
- Closed-lost opportunities: 4
- Closed-won rate: 60%
- Highest overdue rate: Rahul and Meera, at 75% each.
- Lowest overdue rate: Arjun, at 25%.

### 4. Pipeline Stage Analysis
- Negotiation: ₹45 lakh (42.53%)
- Proposal: ₹35 lakh (33.08%)
- Needs Analysis: ₹17.10 lakh (16.16%)
- Qualification: ₹8.70 lakh (8.22%)

### 5. Priority Investigation
OPP030, assigned to Meera, has an opportunity value of ₹9 lakh and is both overdue and stale.

Recommended action: investigate customer engagement, procurement, budget and decision-making blockers. Update CRM and forecast expectations only after obtaining supporting evidence.

### 6. Data Validation
- 30 records successfully imported.
- 30 unique Opportunity IDs.
- Zero duplicate records.
- Expected missing dates validated against opportunity status.
- Date columns converted to datetime.
- Python findings reconciled with SQL and Excel results.

### 7. Conclusion
Python and Pandas were used to inspect, validate, clean and analyse the Nexora sales dataset. The analysis identified follow-up delays, stale opportunities, pipeline concentration and representative-level risks. The findings provide a basis for evidence-driven sales management recommendations.

In [33]:
python_summary.to_csv("Nexora_Python_Findings.csv", index=False)

print("Python findings exported successfully!")

Python findings exported successfully!
